# 21.2 Spark SQL 与查询优化 / Spark SQL & Query Optimization (Catalyst, Joins, Tuning)

**中文**:上一节我们用 RDD 手动写 `map`/`filter`——**你怎么写,Spark 就怎么执行**,写得笨就跑得慢。**Spark SQL / DataFrame API** 是革命性的一步:你用**声明式**的方式说"我要什么"(SQL 或 DataFrame 算子),而**怎么高效执行交给 Catalyst 优化器**——它会自动重写你的查询,让笨写法也变快。这是 Spark 在生产中被大规模使用的关键,也是面试高频考点(优化器、Join 策略、数据倾斜)。本节从零实现一个 **MiniSQL 优化器**,亲手演示两个最重要的优化——**谓词下推(predicate pushdown)** 和 **广播 Join(broadcast join)**,再用一个**真实的分析型优化器(DuckDB)** 的 `EXPLAIN` 印证:工业引擎做的是同一件事。
**English**: Last section we hand-wrote `map`/`filter` with RDDs — **Spark executes exactly what you write**, so clumsy code runs slow. **Spark SQL / the DataFrame API** is a revolutionary step: you say **declaratively** "what I want" (SQL or DataFrame operators) and **leave how to execute it efficiently to the Catalyst optimizer** — which automatically rewrites your query so even clumsy code runs fast. This is key to Spark's large-scale production use and a frequent interview topic (optimizer, join strategies, data skew). This section builds a **MiniSQL optimizer from scratch** to demonstrate the two most important optimizations — **predicate pushdown** and **broadcast join** — then confirms with a **real analytical optimizer (DuckDB)**'s `EXPLAIN` that industrial engines do the same thing.

---

**中文**:**Catalyst 优化器的四步流水线**(面试常问):
**English**: **Catalyst's four-stage pipeline** (a common interview question):
1. **中文**:**未解析逻辑计划(unresolved logical plan)**:解析 SQL/DataFrame 得到的原始算子树。
   **Unresolved logical plan**: the raw operator tree parsed from SQL/DataFrame.
2. **中文**:**解析 + 分析(analysis)**:绑定表名、列名、类型(查元数据目录)。
   **Analysis**: bind table names, column names, types (via the metadata catalog).
3. **中文**:**逻辑优化(logical optimization)**:基于规则重写——**谓词下推**(把 filter 尽量往数据源推)、**列裁剪**(只读用到的列)、常量折叠、**Join 重排**。
   **Logical optimization**: rule-based rewrites — **predicate pushdown** (push filters toward the source), **column pruning** (read only needed columns), constant folding, **join reordering**.
4. **中文**:**物理计划(physical planning)**:基于代价选择**具体执行策略**——尤其是 **Join 用哪种算法**(广播/排序合并/哈希),再生成 Java 字节码(Tungsten 全阶段代码生成)。
   **Physical planning**: cost-based selection of **concrete execution strategies** — especially **which join algorithm** (broadcast / sort-merge / hash), then generate Java bytecode (Tungsten whole-stage codegen).

**中文**:**核心洞察**:因为你写的是**声明式**的"要什么"而非"怎么做",引擎才有自由**重写**成高效计划。同样一句 SQL,Catalyst 能把它优化成比你手写 RDD 快几倍的物理计划。
**English**: **Core insight**: because you write **declaratively** "what" rather than "how," the engine is free to **rewrite** into an efficient plan. The same SQL can be optimized by Catalyst into a physical plan several times faster than your hand-written RDD.

> 💡 **面试速查 / Interview cheat-sheet（★★★ Spark 调优必考）**
> **中文**:**Catalyst 优化器**:未解析逻辑计划→分析(绑定 schema)→逻辑优化(**谓词下推/列裁剪/常量折叠/Join 重排**, 基于规则)→物理计划(基于代价选 Join 策略 + Tungsten 代码生成)。**三种 Join 策略**:①**Broadcast Hash Join**(小表广播到每个执行器, 大表不 shuffle, **最快**, 默认小于 `spark.sql.autoBroadcastJoinThreshold`≈10MB 时用)；②**Sort-Merge Join**(两大表各自按 key 排序+shuffle, 大表 join 默认)；③Shuffle Hash Join。**核心调优**:①**减 Shuffle**(broadcast 小表、reduceByKey);②**数据倾斜**(某 key 数据量爆炸→个别 task 拖死全局→salting 加盐打散 / AQE 自动处理倾斜);③**分区裁剪+列裁剪**(Parquet 分区表, 只读需要的);④**AQE(Adaptive Query Execution)** 运行时根据实际数据量动态调分区数/Join 策略/合并小分区;⑤`cache()` 复用、避免 `collect()`、`repartition/coalesce` 控分区。**看 Spark UI** 找 shuffle/spill/skew。面试金句:*"Spark SQL 靠 Catalyst 把声明式查询优化成高效物理计划——谓词下推、列裁剪减少读入数据, 物理阶段按代价选 Join 策略; 大表 join 用 sort-merge、有小表就 broadcast 避免 shuffle; 最大痛点是 shuffle 和数据倾斜, 用 broadcast/salting/AQE 解决。"*
> **English**: **Catalyst optimizer**: unresolved logical plan → analysis (bind schema) → logical optimization (**predicate pushdown / column pruning / constant folding / join reordering**, rule-based) → physical planning (cost-based join strategy + Tungsten codegen). **Three join strategies**: ① **Broadcast Hash Join** (broadcast the small table to every executor, big table not shuffled, **fastest**, default when smaller than `spark.sql.autoBroadcastJoinThreshold`≈10MB); ② **Sort-Merge Join** (both big tables sorted by key + shuffled, default for big-big joins); ③ Shuffle Hash Join. **Core tuning**: ① **reduce Shuffle** (broadcast small tables, reduceByKey); ② **data skew** (one key explodes → a few tasks stall the whole job → salting to spread / AQE auto-handles skew); ③ **partition + column pruning** (Parquet partitioned tables, read only what's needed); ④ **AQE (Adaptive Query Execution)** dynamically adjusts partition counts / join strategy / coalesces small partitions at runtime based on actual data; ⑤ `cache()` for reuse, avoid `collect()`, `repartition/coalesce` to control partitions. **Read the Spark UI** to find shuffle/spill/skew. Interview line: *"Spark SQL uses Catalyst to optimize declarative queries into efficient physical plans — predicate pushdown and column pruning cut data read, and the physical phase picks a join strategy by cost; big-big joins use sort-merge, a small table triggers broadcast to avoid shuffle; the biggest pains are shuffle and data skew, solved with broadcast/salting/AQE."*


In [ ]:

# ============================================================
# 从零实现 MiniSQL:演示谓词下推 + 广播/Shuffle Join / MiniSQL: pushdown + broadcast vs shuffle join
# 中文:两张"分布式表"——orders(2万行, 大)和 users(50行, 小)。我们统计两种关键代价:
#      rows_scanned(读入行数)和 rows_shuffled(跨网络重排的行数, 分布式最贵的开销)。
# English: two "distributed tables" — orders (20k rows, big) and users (50 rows, small). We track two key costs:
#      rows_scanned and rows_shuffled (rows reshuffled across the network — the most expensive distributed cost).
# ============================================================
import random
from collections import defaultdict
random.seed(0)
def partition(rows, nparts=8): return [rows[i::nparts] for i in range(nparts)]   # 切成分区 / split into partitions
orders=partition([{"user_id":random.randint(1,50),"amount":random.randint(1,500),
                   "country":random.choice(["US","UK","CN","IN"])} for _ in range(20000)])
users =partition([{"user_id":u,"vip":u%7==0} for u in range(1,51)])              # 小维表 / small dimension table

METRIC={"scanned":0,"shuffled":0}
def scan(parts):
    for p in parts:
        for _ in p: METRIC["scanned"]+=1                                          # 记录读入行数 / count rows read
    return parts
def where(parts,pred): return [[r for r in p if pred(r)] for p in parts]          # filter(窄依赖)/ narrow
def join_shuffle(left,right,key):     # Sort-Merge/Shuffle Join:两边都按 key 跨网络重排 / both sides reshuffled by key
    rb=defaultdict(list)
    for p in right:
        for r in p: METRIC["shuffled"]+=1; rb[r[key]].append(r)                   # 右表 shuffle / right shuffled
    out=[]
    for p in left:
        for l in p:
            METRIC["shuffled"]+=1                                                 # 左表也 shuffle / left shuffled too
            out+=[{**l,**r} for r in rb.get(l[key],[])]
    return [out]
def join_broadcast(left,right,key):   # Broadcast Join:小表复制到每个执行器, 大表原地不动=零 shuffle / small table broadcast, big stays
    rmap=defaultdict(list)
    for p in right:
        for r in p: rmap[r[key]].append(r)                                        # 广播副本(小)/ broadcast copy (small)
    out=[]
    for p in left:                                                                # 大表不 shuffle / big table not shuffled
        for l in p: out+=[{**l,**r} for r in rmap.get(l[key],[])]
    return [out]

# --- 计划 A(未优化):扫全表 → shuffle join → 最后才 filter country=US ---
METRIC.update(scanned=0,shuffled=0)
a=scan(orders); a=join_shuffle(a,users,"user_id"); a=where(a,lambda r:r["country"]=="US")
planA=(METRIC["scanned"],METRIC["shuffled"],sum(len(p) for p in a))
# --- 计划 B(优化):谓词下推(先 filter)→ broadcast join(users 很小)---
METRIC.update(scanned=0,shuffled=0)
b=scan(orders); b=where(b,lambda r:r["country"]=="US"); b=join_broadcast(b,users,"user_id")
planB=(METRIC["scanned"],METRIC["shuffled"],sum(len(p) for p in b))
print(f"计划A 未优化(filter 后置 + shuffle join): 扫描 {planA[0]:>6} 行, 跨网络 shuffle {planA[1]:>6} 行, 结果 {planA[2]} 行")
print(f"计划B 优化(谓词下推 + broadcast join):   扫描 {planB[0]:>6} 行, 跨网络 shuffle {planB[1]:>6} 行, 结果 {planB[2]} 行")
print(f"\n两计划结果完全相同({planA[2]}={planB[2]}行), 但优化后 Shuffle 从 {planA[1]} 降到 {planB[1]}——这就是 Catalyst 的价值")


In [ ]:

# ============================================================
# 用真实分析型优化器(DuckDB)印证:EXPLAIN 展示谓词下推 + 列裁剪 / real optimizer confirms it
# 中文:DuckDB 是一个真实的、生产级的分析 SQL 引擎, 有和 Catalyst 一样的优化器。看它怎么优化同一个查询。
# English: DuckDB is a real, production-grade analytical SQL engine with a Catalyst-like optimizer. See how it optimizes.
# ============================================================
import duckdb
con=duckdb.connect()
con.execute("CREATE TABLE orders AS SELECT (i%50)+1 AS user_id, (i%500)+1 AS amount, "
            "['US','UK','CN','IN'][(i%4)+1] AS country FROM range(20000) t(i)")
con.execute("CREATE TABLE users AS SELECT i+1 AS user_id, (i%7=0) AS vip FROM range(50) t(i)")
plan=con.execute("""EXPLAIN SELECT o.user_id, o.amount
                    FROM orders o JOIN users u ON o.user_id=u.user_id
                    WHERE o.country='US'""").fetchall()[0][1]
# 只截取关键部分展示 / show the key part
print("DuckDB 的物理计划(真实优化器)/ DuckDB physical plan (real optimizer):")
print("\n".join(l for l in plan.splitlines() if any(k in l for k in
      ["PROJECTION","HASH_JOIN","SEQ_SCAN","Filters","country","Projections","rows"]))[:900])
print("\n观察:①country='US' 的 Filter 被下推进 orders 的扫描(不是 join 后才过滤);"
      "\n     ②只投影 user_id/amount(列裁剪, 不读 country 之外多余列);③join 选了 HASH_JOIN。")
print("→ 和我们 MiniSQL 手动做的谓词下推是同一个思想, 只是 DuckDB/Spark 自动完成。")


In [ ]:

# ============================================================
# 可视化:优化前后的代价对比 / visualize cost before/after optimization
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① Shuffle 代价 / shuffle cost
labels=["计划A\n未优化","计划B\n谓词下推+broadcast"]
ax[0].bar(labels,[planA[1],planB[1]],color=["#C44E52","#55A868"])
for i,v in enumerate([planA[1],planB[1]]): ax[0].text(i,v+300,f"{v:,}",ha="center",fontsize=11,weight="bold")
ax[0].set_ylabel("跨网络 Shuffle 的行数(越低越好)"); ax[0].set_title("Catalyst 优化把 Shuffle 从 2万+ 降到 0")
# ② 逻辑计划重写示意 / logical plan rewrite
ax[1].axis("off"); ax[1].set_title("谓词下推:把 Filter 推到扫描处",fontsize=12,weight="bold")
ax[1].text(0.02,0.9,"优化前 / before:",fontsize=10,weight="bold",color="#C44E52",transform=ax[1].transAxes)
ax[1].text(0.05,0.62,"Filter(country=US)\n   └─ Join\n        ├─ Scan orders (读全部2万行)\n        └─ Scan users",
           fontsize=9,family="monospace",transform=ax[1].transAxes)
ax[1].text(0.02,0.42,"优化后 / after (谓词下推):",fontsize=10,weight="bold",color="#55A868",transform=ax[1].transAxes)
ax[1].text(0.05,0.1,"Join (broadcast users)\n   ├─ Scan orders + Filter(country=US) ← 下推!\n   └─ Broadcast Scan users",
           fontsize=9,family="monospace",transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/big02_viz.png",dpi=80); plt.show()
print("Filter 越早执行, 后续 join/shuffle 处理的数据越少; broadcast 小表让大表完全不用 shuffle")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **声明式是"自动优化"的前提**:上一节手写 RDD,你怎么写就怎么跑;这一节改用声明式 SQL/DataFrame,引擎才有自由**重写**你的查询。同样的最终结果(5008 行),Catalyst 通过谓词下推 + 广播 Join,把跨网络 Shuffle 从 2 万多行降到 **0**——这不是小优化,在真实 TB 级数据上,一次多余的 Shuffle 可能意味着几小时和几分钟的差距。**核心教训:能用 DataFrame/SQL 就别用 RDD**,把优化交给比你更懂执行细节的引擎。
2. **Join 策略是 Spark 调优的第一战场**:两张大表 join 必须各自 Shuffle(sort-merge join),这很贵;但只要有一边足够小(能装进内存,默认 <10MB),就该用 **Broadcast Join**——把小表复制到每个执行器,大表原地做本地哈希 join,**完全避免大表的 Shuffle**。我们的 MiniSQL 里 `join_broadcast` 的 Shuffle 计数为 0,正是这个道理。实战中一个极常见的加速,就是给 join 加 `broadcast(small_df)` 提示;而忘了广播、让两张表都 Shuffle,是新手最常见的性能坑。
3. **诚实的复杂性:优化器很强,但不是万能,数据倾斜会击穿一切**。①**优化器基于统计估计**——如果表统计信息过时/缺失,它可能选错 Join 策略(比如没广播本该广播的小表);生产中要 `ANALYZE`/维护统计。②**数据倾斜(data skew)是最难的问题**:如果某个 key(比如 `user_id=NULL` 或某个超级大客户)占了一半数据,那么负责这个 key 的那**一个** task 会处理海量数据、其他 task 早就闲着——整个作业被这一个慢 task 拖死,还常常 OOM。优化器帮不了你,得靠**加盐(salting,给热点 key 加随机后缀打散)** 或 Spark 3 的 **AQE(自适应查询执行,运行时检测倾斜并自动拆分)**。③**AQE 是现代 Spark 的一大进步**——它在运行时根据**真实**数据量动态调整(合并小分区、切换 Join 策略、处理倾斜),弥补了静态优化器"猜错"的问题。**记住:优化器解决"写得笨",但解决不了"数据本身长得歪"——倾斜要靠人和 AQE。**

**English**:
1. **Declarativeness is the prerequisite for "automatic optimization"**: last section's hand-written RDD runs exactly as written; switching to declarative SQL/DataFrame frees the engine to **rewrite** your query. For the same final result (5008 rows), Catalyst's predicate pushdown + broadcast join cut cross-network Shuffle from 20k+ rows to **0** — not a minor tweak: on real TB-scale data, one extra Shuffle can mean hours vs minutes. **Core lesson: use DataFrame/SQL instead of RDD whenever possible**, leaving optimization to an engine that knows execution details better than you.
2. **Join strategy is the first battlefield of Spark tuning**: joining two big tables requires each to Shuffle (sort-merge join), which is expensive; but if either side is small enough (fits in memory, default <10MB), use a **Broadcast Join** — copy the small table to every executor and do a local hash join on the big table in place, **completely avoiding the big table's Shuffle**. Our MiniSQL's `join_broadcast` Shuffle count of 0 is exactly this. A very common real speedup is adding a `broadcast(small_df)` hint; forgetting to broadcast and letting both tables Shuffle is the most common beginner performance trap.
3. **Honest complexity: the optimizer is strong but not omnipotent, and data skew defeats everything**. ① **The optimizer relies on statistical estimates** — if table stats are stale/missing, it may pick the wrong join strategy (e.g., not broadcasting a small table it should); production needs `ANALYZE`/maintained stats. ② **Data skew is the hardest problem**: if one key (say `user_id=NULL` or a single huge customer) holds half the data, the **single** task for that key processes a massive amount while others idle — the whole job is stalled by that one slow task and often OOMs. The optimizer can't help; you need **salting (add a random suffix to hot keys to spread them)** or Spark 3's **AQE (Adaptive Query Execution, detects skew at runtime and auto-splits)**. ③ **AQE is a major modern-Spark advance** — it adjusts dynamically at runtime based on **actual** data (coalescing small partitions, switching join strategies, handling skew), compensating for a static optimizer's mis-guesses. **Remember: the optimizer fixes "clumsy code" but not "skewed data" — skew is on you and AQE.**

> 💼 **实战视角 / Practical angle**
> **中文**:Spark SQL 调优实战:①**优先 DataFrame/SQL API**(享受 Catalyst), 少写 RDD/UDF(UDF 是优化黑盒);②**广播小表** `F.broadcast(dim_df)` 或调 `autoBroadcastJoinThreshold`;③**开 AQE**(`spark.sql.adaptive.enabled=true`, Spark3 默认开)自动处理倾斜和小分区;④**治数据倾斜**:定位热点 key(看 Spark UI 里某 task 特别慢)→ salting / 拆分 / 广播;⑤**用 Parquet 分区表** + 分区裁剪 + 列裁剪(只读需要的分区和列);⑥`cache()` 复用中间结果、避免 `collect()` 大数据回 Driver。**排查工具**:Spark UI(看 stage 的 shuffle read/write、task 时间分布=倾斜、spill=内存不足)、`df.explain()` 看物理计划。面试金句:*"Spark SQL 调优核心是减 shuffle 和治倾斜:能广播就广播小表避免 sort-merge shuffle、开 AQE 让引擎运行时自适应、用 Parquet 分区+列裁剪少读数据、给倾斜的热点 key 加盐; 用 df.explain() 和 Spark UI 定位瓶颈。"*
> **English**: Spark SQL tuning in practice: ① **prefer the DataFrame/SQL API** (enjoy Catalyst), minimize RDD/UDFs (UDFs are optimization black boxes); ② **broadcast small tables** `F.broadcast(dim_df)` or tune `autoBroadcastJoinThreshold`; ③ **enable AQE** (`spark.sql.adaptive.enabled=true`, default on in Spark 3) to auto-handle skew and small partitions; ④ **fix data skew**: locate hot keys (one task far slower in the Spark UI) → salting / splitting / broadcasting; ⑤ **use Parquet partitioned tables** + partition pruning + column pruning (read only needed partitions and columns); ⑥ `cache()` to reuse intermediates, avoid `collect()` bringing big data to the Driver. **Debug tools**: Spark UI (stage shuffle read/write, task-time distribution = skew, spill = OOM), `df.explain()` for the physical plan. Interview line: *"Spark SQL tuning centers on reducing shuffle and fixing skew: broadcast small tables to avoid sort-merge shuffle, enable AQE for runtime adaptivity, use Parquet partitioning + column pruning to read less, and salt skewed hot keys; use df.explain() and the Spark UI to locate bottlenecks."*

---
### 小结 / Summary
- **中文**:声明式 SQL/DataFrame → Catalyst 自动优化(谓词下推、列裁剪、Join 重排、代码生成), 远优于手写 RDD。
- **English**: Declarative SQL/DataFrame → Catalyst auto-optimizes (predicate pushdown, column pruning, join reordering, codegen), far better than hand-written RDD.
- **中文**:Join 策略:大表 join 用 sort-merge(双方 shuffle), 有小表用 broadcast(零大表 shuffle, 最快)。
- **English**: Join strategies: big-big uses sort-merge (both shuffle), a small table triggers broadcast (zero big-table shuffle, fastest).
- **中文**:最大痛点=Shuffle 和数据倾斜; 用 broadcast/salting/AQE/分区列裁剪解决, 靠 Spark UI + explain() 定位。
- **English**: Biggest pains = Shuffle and data skew; solved with broadcast/salting/AQE/partition-column pruning, located via Spark UI + explain().
